In [9]:
!pip install -q imagehash

In [ ]:
from pathlib import Path
import re
import hashlib
import pandas as pd
from PIL import Image, ImageOps
import imagehash

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")
SELECTED_CROPS = {"corn", "tomato", "potato", "bell_pepper", "soybean"}
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png"}

def find_dataset_root():
    candidates = []

    for path in INPUT_ROOT.rglob("*"):
        if path.is_dir() and (path / "train").is_dir() and (path / "val").is_dir():
            candidates.append(path)

    if not candidates:
        raise FileNotFoundError(
            "Could not find a folder containing both train/ and val/. "
            "Run: !find /kaggle/input -maxdepth 4 -type d"
        )

    return candidates[0]

def normalize_name(text):
    text = text.lower().strip()
    text = text.replace("_(maize)", "")
    text = text.replace("pepper,_bell", "bell_pepper")
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return re.sub(r"_+", "_", text).strip("_")

def parse_folder_name(folder_name):
    if "___" in folder_name:
        crop_raw, disease_raw = folder_name.split("___", 1)
    else:
        crop_raw, disease_raw = folder_name, "unknown"

    crop = normalize_name(crop_raw)
    disease = normalize_name(disease_raw)
    is_healthy = int(disease == "healthy")

    return crop, disease, is_healthy

def get_sha256(file_path):
    hash_object = hashlib.sha256()

    with open(file_path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            hash_object.update(block)

    return hash_object.hexdigest()

def get_image_info(file_path):
    try:
        with Image.open(file_path) as image:
            image = ImageOps.exif_transpose(image)
            width, height = image.size
            phash = str(imagehash.phash(image.convert("RGB")))

        return width, height, phash, 0

    except Exception:
        return None, None, None, 1

DATASET_ROOT = find_dataset_root()

print("Dataset root found:")
print(DATASET_ROOT)

records = []

for split in ["train", "val"]:
    split_path = DATASET_ROOT / split

    for class_folder in sorted(split_path.iterdir()):
        if not class_folder.is_dir():
            continue

        crop, disease, is_healthy = parse_folder_name(class_folder.name)

        if crop not in SELECTED_CROPS:
            continue

        for image_path in sorted(class_folder.rglob("*")):
            if not image_path.is_file():
                continue

            if image_path.suffix.lower() not in VALID_EXTENSIONS:
                continue

            width, height, phash, is_corrupt = get_image_info(image_path)

            records.append(
                {
                    "image_path": str(image_path),
                    "relative_path": str(image_path.relative_to(DATASET_ROOT)),
                    "split": split,
                    "source_class_folder": class_folder.name,
                    "crop": crop,
                    "disease": disease,
                    "label": f"{crop}__{disease}",
                    "is_healthy": is_healthy,
                    "file_name": image_path.name,
                    "file_extension": image_path.suffix.lower(),
                    "file_size_kb": round(image_path.stat().st_size / 1024, 2),
                    "width": width,
                    "height": height,
                    "sha256": None if is_corrupt else get_sha256(image_path),
                    "phash": phash,
                    "is_corrupt": is_corrupt
                }
            )

manifest = pd.DataFrame(records)

manifest_path = OUTPUT_DIR / "plantvillage_selected_manifest.csv"
summary_path = OUTPUT_DIR / "plantvillage_selected_summary.csv"

manifest.to_csv(manifest_path, index=False)

summary = (
    manifest
    .groupby(["split", "crop", "disease", "label"], as_index=False)
    .agg(
        image_count=("image_path", "count"),
        corrupt_images=("is_corrupt", "sum"),
        mean_width=("width", "mean"),
        mean_height=("height", "mean")
    )
)

summary.to_csv(summary_path, index=False)

print("\nManifest created successfully")
print(f"Images: {len(manifest)}")
print(f"Classes: {manifest['label'].nunique()}")
print(f"Corrupt images: {manifest['is_corrupt'].sum()}")
print(f"\nSaved: {manifest_path}")
print(f"Saved: {summary_path}")

display(
    manifest
    .groupby(["split", "label"])
    .size()
    .unstack(fill_value=0)
)